## Imports

In [ ]:
from __future__ import annotations

import sys
import json
import shutil
import hashlib
import platform
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

## Config

In [ ]:
def load_cfg_fallback() -> dict:
    if "CFG" in globals() and isinstance(globals()["CFG"], dict):
        return globals()["CFG"]

    candidates = [
        Path("/mnt/data/config_snapshot.json"),
        Path("/content/reports/config_snapshot.json"),
        Path("/content/config_snapshot.json"),
        Path("/mnt/data/config.json"),
        Path("/content/config.json"),
        Path("./config.json"),
    ]

    for p in candidates:
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                cfg = json.load(f)
            print(f"Loaded CFG from: {p}")
            return cfg

    raise FileNotFoundError(
        "CFG not found in memory and no config_snapshot.json/config.json found in expected locations."
    )


CFG = load_cfg_fallback()

paths = CFG["paths"]
output_dir = Path(paths["output_dir"])
tables_dir_cfg = Path(paths["tables_dir"])
figures_dir_cfg = Path(paths["figures_dir"])
reports_dir_cfg = Path(paths["reports_dir"])
models_dir_cfg = Path(paths["models_dir"])
intermediate_dir_cfg = Path(paths["intermediate_dir"])
supplementary_dir_cfg = Path(paths.get("supplementary_dir", str(tables_dir_cfg)))
qc_dir_cfg = Path(paths.get("qc_dir", output_dir / "qc"))

print("Configured output_dir:", output_dir)
print("Configured tables_dir:", tables_dir_cfg)
print("Configured figures_dir:", figures_dir_cfg)
print("Configured reports_dir:", reports_dir_cfg)
print("Configured models_dir:", models_dir_cfg)
print("Configured intermediate_dir:", intermediate_dir_cfg)
print("Configured supplementary_dir:", supplementary_dir_cfg)
print("Configured qc_dir:", qc_dir_cfg)

Loaded CFG from: /content/config_snapshot.json
Configured output_dir: /content/outputs
Configured tables_dir: /content/outputs/tables
Configured figures_dir: /content/outputs/figures
Configured reports_dir: /content/reports
Configured models_dir: /content/outputs/models
Configured intermediate_dir: /content/outputs/intermediate
Configured supplementary_dir: /content/outputs/supplementary
Configured qc_dir: /content/outputs/qc


## Release / export directories

In [ ]:

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
project_name = CFG.get("project", {}).get("name", "project")
version = CFG.get("project", {}).get("version", "0.0.0")

export_work_dir = output_dir / "s11_export_work"
release_root = output_dir / "release"
bundle_dir = release_root / f"{project_name}_v{version}_{timestamp}_analysis_bundle"

bundle_tables = bundle_dir / "tables"
bundle_figures = bundle_dir / "figures"
bundle_reports = bundle_dir / "reports"
bundle_data = bundle_dir / "data"
bundle_models = bundle_dir / "models"
bundle_meta = bundle_dir / "meta"

for p in [
    export_work_dir,
    release_root,
    bundle_dir,
    bundle_tables,
    bundle_figures,
    bundle_reports,
    bundle_data,
    bundle_models,
    bundle_meta,
]:
    p.mkdir(parents=True, exist_ok=True)

print("S11 work dir:", export_work_dir)
print("Bundle dir:", bundle_dir)

S11 work dir: /content/outputs/s11_export_work
Bundle dir: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle


## Helpers

In [ ]:

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def safe_copy(src: Path, dst_dir: Path) -> Optional[Path]:
    if not src.exists():
        return None
    dst = dst_dir / src.name
    shutil.copy2(src, dst)
    return dst


def try_read_csv(path: Path) -> Optional[pd.DataFrame]:
    if not path.exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] Could not read CSV {path.name}: {repr(e)}")
        return None


def pick_first_existing(candidates: List[Path]) -> Optional[Path]:
    for p in candidates:
        if p.exists():
            return p
    return None


def standardize_result_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    ren = {}
    for c in out.columns:
        cl = str(c).lower().strip()

        if cl in {"or", "odds_ratio"}:
            ren[c] = "OR"
        elif cl in {"ci_low", "lower_ci", "ci_lower"}:
            ren[c] = "CI_low"
        elif cl in {"ci_high", "upper_ci", "ci_upper"}:
            ren[c] = "CI_high"
        elif cl in {"p", "pvalue", "p_value"}:
            ren[c] = "p"
        elif cl in {"term", "predictor", "variable"}:
            ren[c] = "term"
        elif cl in {"endpoint", "outcome", "dependent"}:
            ren[c] = "endpoint"
        elif cl in {"exposure"}:
            ren[c] = "exposure"
        elif cl in {"model", "model_set"}:
            ren[c] = "model_set"

    if ren:
        out = out.rename(columns=ren)

    return out


def collect_unique_files(patterns_by_dir: List[Tuple[Path, str]]) -> List[Path]:
    found: List[Path] = []
    for base_dir, pattern in patterns_by_dir:
        if base_dir.exists() and base_dir.is_dir():
            found.extend(base_dir.glob(pattern))
    return sorted(set(found))


## Controlled source discovery

In [ ]:

def existing_dirs(candidates: List[Path]) -> List[Path]:
    out = []
    seen = set()
    for p in candidates:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if p.exists() and p.is_dir() and str(rp) not in seen:
            out.append(p)
            seen.add(str(rp))
    return out


search_root_candidates = existing_dirs([
    tables_dir_cfg,
    supplementary_dir_cfg,
    qc_dir_cfg,
    figures_dir_cfg,
    reports_dir_cfg,
    models_dir_cfg,
    output_dir,
    intermediate_dir_cfg,
    Path("/content/outputs"),
    Path("/content"),
    Path("/mnt/data"),
])

print("Search roots:")
for p in search_root_candidates:
    print(" -", p)

table_sources = []
figure_sources = []
report_sources = []
model_sources = []

for root in search_root_candidates:
    table_sources.extend([
        (root, "*.csv"),
    ])
    figure_sources.extend([
        (root, "*.png"),
        (root, "*.pdf"),
        (root, "*.svg"),
    ])
    report_sources.extend([
        (root, "*.json"),
        (root, "*.txt"),
        (root, "*.md"),
        (root, "*.html"),
    ])
    model_sources.extend([
        (root, "*.pkl"),
        (root, "*.joblib"),
    ])

table_files = collect_unique_files(table_sources)
fig_files = collect_unique_files(figure_sources)
rep_files = collect_unique_files(report_sources)
model_files = collect_unique_files(model_sources)

print("Tables found:", len(table_files))
print("Figures found:", len(fig_files))
print("Reports found:", len(rep_files))
print("Models found:", len(model_files))

print("\nSample tables:")
for p in table_files[:10]:
    print(" -", p)

print("\nSample figures:")
for p in fig_files[:10]:
    print(" -", p)

print("\nSample reports:")
for p in rep_files[:10]:
    print(" -", p)

if len(table_files) == 0:
    raise RuntimeError(
        "No CSV tables found in configured or fallback source locations."
    )

Search roots:
 - /content/outputs
 - /content
Tables found: 26
Figures found: 10
Reports found: 9
Models found: 0

Sample tables:
 - /content/MANIFEST_sha256.csv
 - /content/MASTER_results_compiled.csv
 - /content/MASTER_supplement_compiled.csv
 - /content/Table1_TAI_A_baseline.csv
 - /content/Table1_event_counts_by_TAI_A.csv
 - /content/Table3_secondary_models.csv
 - /content/Table4_secondary_models_continuous_companion.csv
 - /content/TableS10_HL.csv
 - /content/TableS10_VIF.csv
 - /content/TableS10_age_spline_LR_primary.csv

Sample figures:
 - /content/F1_distributions_TAI_A.png
 - /content/F2_androgens_by_TAI_A.png
 - /content/F3_missingness_barplot.png
 - /content/F6_calibration_non_hdl.png
 - /content/F6_calibration_ogtt120.png
 - /content/F7_forest_secondary_OR.png
 - /content/F_S10_calibration_ep_non_hdl.png
 - /content/F_S10_calibration_ep_ogtt120.png
 - /content/F_S10_calibration_ep_primary.png
 - /content/F_S10_cooks_primary.png

Sample reports:
 - /content/S10_diagnostics_n

## Data candidates - controlled, but includes config/meta fallbacks

In [ ]:

data_candidates = [
    output_dir / "pcos_base.csv",
    output_dir / "pcos_base.parquet",
    output_dir / "pcos_qc.csv",
    output_dir / "pcos_qc.parquet",
    output_dir / "pcos_analysis.csv",
    output_dir / "pcos_analysis.parquet",
    intermediate_dir_cfg / "pcos_base.csv",
    intermediate_dir_cfg / "pcos_base.parquet",
    intermediate_dir_cfg / "pcos_qc.csv",
    intermediate_dir_cfg / "pcos_qc.parquet",
    intermediate_dir_cfg / "pcos_analysis.csv",
    intermediate_dir_cfg / "pcos_analysis.parquet",
    reports_dir_cfg / "config_snapshot.json",
    Path("/mnt/data/config_snapshot.json"),
    Path("/mnt/data/config.json"),
    Path("/mnt/data/column_mapping.json"),
]

data_files = sorted({p for p in data_candidates if p.exists()})

print("Data candidates found:", len(data_files))

print("\n=== Source summary before copying ===")
print("table_files:", len(table_files))
print("fig_files:", len(fig_files))
print("rep_files:", len(rep_files))
print("data_files:", len(data_files))
print("model_files:", len(model_files))

Data candidates found: 0

=== Source summary before copying ===
table_files: 26
fig_files: 10
rep_files: 9
data_files: 0
model_files: 0


## Copy source artifacts into bundle

In [ ]:

copied: List[Path] = []

for f in table_files:
    dst = safe_copy(f, bundle_tables)
    if dst:
        copied.append(dst)

for f in fig_files:
    dst = safe_copy(f, bundle_figures)
    if dst:
        copied.append(dst)

for f in rep_files:
    dst = safe_copy(f, bundle_reports)
    if dst:
        copied.append(dst)

for f in data_files:
    if f.name in {"config.json", "config_snapshot.json", "column_mapping.json"}:
        dst = safe_copy(f, bundle_meta)
    else:
        dst = safe_copy(f, bundle_data)
    if dst:
        copied.append(dst)

for f in model_files:
    dst = safe_copy(f, bundle_models)
    if dst:
        copied.append(dst)

print("Copied files:", len(copied))


Copied files: 45


## Build MASTER_results_compiled.csv

In [ ]:

master_parts = []

# Primary model results (S05)
primary_candidates = [
    bundle_tables / "Table2_primary_model.csv",
    bundle_tables / "Table1_primary_model.csv",
]
t_primary = pick_first_existing(primary_candidates)
if t_primary is not None:
    df_primary = try_read_csv(t_primary)
    if df_primary is not None:
        dfp = standardize_result_table(df_primary)
        dfp["source_table"] = t_primary.stem
        master_parts.append(dfp)

# Secondary models (S06)
t_secondary = bundle_tables / "Table3_secondary_models.csv"
df_secondary = try_read_csv(t_secondary)
if df_secondary is not None:
    dfs = standardize_result_table(df_secondary)
    dfs["source_table"] = "Table3_secondary_models"
    master_parts.append(dfs)

# Sensitivity (S08)
t_sens = bundle_tables / "TableS1_sensitivity.csv"
df_sens = try_read_csv(t_sens)
if df_sens is not None:
    dfa = standardize_result_table(df_sens)
    dfa["source_table"] = "TableS1_sensitivity"
    master_parts.append(dfa)

# E-values (S09)
t_eval = bundle_tables / "TableS2_Evalues.csv"
df_eval = try_read_csv(t_eval)
if df_eval is not None:
    dfe = standardize_result_table(df_eval)
    dfe["source_table"] = "TableS2_Evalues"
    master_parts.append(dfe)

# Diagnostics (S10)
t_diag = bundle_tables / "TableS10_model_diagnostics.csv"
df_diag = try_read_csv(t_diag)
if df_diag is not None:
    dfd = standardize_result_table(df_diag)
    dfd["source_table"] = "TableS10_model_diagnostics"
    master_parts.append(dfd)

# Cohort/event integrity (S10)
t_cohort = bundle_tables / "TableS10_cohort_integrity_and_events.csv"
df_cohort = try_read_csv(t_cohort)
if df_cohort is not None:
    dfc = standardize_result_table(df_cohort)
    dfc["source_table"] = "TableS10_cohort_integrity_and_events"
    master_parts.append(dfc)

master_results = pd.concat(master_parts, ignore_index=True) if master_parts else pd.DataFrame()

if master_results.empty:
    raise RuntimeError(
        "MASTER_results_compiled.csv is empty. "
        "This indicates that S11 did not find upstream result tables."
    )

out_master = bundle_tables / "MASTER_results_compiled.csv"
master_results.to_csv(out_master, index=False)

print("Saved:", out_master)
print("Rows:", len(master_results))
print(master_results.head(10))

Saved: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle/tables/MASTER_results_compiled.csv
Rows: 12
        term  beta_boot_median  OR_boot_median    CI_low   CI_high  \
0  Intercept         -1.317693        0.267752  0.227871  0.310083   
1      TAI_A          0.086289        1.090121  0.611688  1.763087   
2      age_c          0.088579        1.092621  0.986589  1.213266   
3  Intercept         -1.746599        0.174366  0.144735  0.207860   
4      TAI_A          0.226890        1.254692  0.625181  2.180289   
5      age_c         -0.002350        0.997653  0.901514  1.108962   
6        NaN               NaN             NaN       NaN       NaN   
7        NaN               NaN             NaN       NaN       NaN   
8        NaN               NaN             NaN       NaN       NaN   
9        NaN               NaN             NaN       NaN       NaN   

   n_boot_success  n_boot_nonconverged  n_boot_failed_or_skipped    endpoint  \
0        

## Build MASTER_supplement_compiled.csv

In [ ]:

supp_parts = []

supp_files = [
    "Table1_TAI_A_baseline.csv",
    "TableS_measured_vs_notmeasured_antiTPO.csv",
    "Table2_primary_model_continuous_companion.csv",
    "Table4_secondary_models_continuous_companion.csv",
    "Table5_group_differences_FDR.csv",
    "TableS_S07_missingness_by_group.csv",
    "TableS2_Evalues.csv",
    "TableS10_VIF.csv",
    "TableS10_HL.csv",
    "TableS10_cohort_integrity_and_events.csv",
    "TableS10_influence_primary.csv",
    "TableS10_age_spline_LR_primary.csv",
]

for fn in supp_files:
    p = bundle_tables / fn
    dfi = try_read_csv(p)
    if dfi is None:
        continue
    dfi = dfi.copy()
    dfi["source_table"] = Path(fn).stem
    supp_parts.append(dfi)

master_supp = pd.concat(supp_parts, ignore_index=True) if supp_parts else pd.DataFrame()

if master_supp.empty:
    print("[WARN] MASTER_supplement_compiled.csv is empty.")

out_supp = bundle_tables / "MASTER_supplement_compiled.csv"
master_supp.to_csv(out_supp, index=False)

print("Saved:", out_supp)
print("Rows:", len(master_supp))
print(master_supp.head(10) if not master_supp.empty else "No supplementary rows.")

Saved: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle/tables/MASTER_supplement_compiled.csv
Rows: 1103
       variable        type               TAI+               TAI−  \
0       N_group        meta                 84                971   
1           age  continuous        22 [21, 23]        22 [20, 23]   
2           tsh  continuous   2.4 [1.46, 3.52]  1.77 [1.27, 2.42]   
3      anti_tpo  continuous    129 [70.9, 261]        9.9 [9, 13]   
4            tg  continuous   75.3 [58.8, 103]     77 [58.9, 105]   
5           hdl  continuous      56.6 [48, 66]  57.8 [48.6, 67.2]   
6            tc  continuous     164 [137, 188]     164 [146, 185]   
7           ldl  continuous   85.5 [68.7, 107]     87.9 [72, 106]   
8  tg_hdl_ratio  continuous  1.24 [0.95, 2.22]    1.29 [0.904, 2]   
9       non_hdl  continuous    103 [81.9, 127]    104 [86.8, 125]   

   n_non_missing_TAI+  n_non_missing_TAI−  n_missing_total    SMD   p_value  \
0               

## STROBE flowchart - generated inside bundle, not upstream figures

In [ ]:

summary_candidates = [
    bundle_tables / "qc_cohort_availability_summary.csv",
    bundle_data / "qc_cohort_availability_summary.csv",
    qc_dir_cfg / "qc_cohort_availability_summary.csv",
    Path("/mnt/data/qc_cohort_availability_summary.csv"),
]

summary_file = pick_first_existing(summary_candidates)

if summary_file is None:
    print("[WARN] qc_cohort_availability_summary.csv not found. STROBE flowchart not generated.")
else:
    summary = pd.read_csv(summary_file)
    summary["stage"] = summary["stage"].astype(str)
    summary["n"] = pd.to_numeric(summary["n"], errors="coerce")

    stage_to_n = dict(zip(summary["stage"], summary["n"]))
    required_stages = [
        "n_total_rows",
        "has_anti_tpo",
        "eligible_primary_minimal",
        "eligible_non_hdl_minimal",
        "eligible_ogtt120_minimal",
    ]

    missing_stages = [s for s in required_stages if s not in stage_to_n]
    if missing_stages:
        print(f"[WARN] STROBE flowchart skipped. Missing stages in summary file: {missing_stages}")
    else:
        n_total = int(stage_to_n["n_total_rows"])
        n_tai_defined = int(stage_to_n["has_anti_tpo"])
        n_primary = int(stage_to_n["eligible_primary_minimal"])
        n_non_hdl = int(stage_to_n["eligible_non_hdl_minimal"])
        n_ogtt120 = int(stage_to_n["eligible_ogtt120_minimal"])

        excluded_missing_anti_tpo = n_total - n_tai_defined
        excluded_primary_after_tai = n_tai_defined - n_primary
        excluded_ogtt_after_tai = n_tai_defined - n_ogtt120

        main_boxes = [
            f"Women with PCOS in source dataset\nn = {n_total}",
            f"TAI status available (anti-TPO measured)\nn = {n_tai_defined}",
            f"Included in primary analysis\nTG/HDL > 3.5 endpoint\nn = {n_primary}",
            f"Included in secondary analysis\nnon-HDL endpoint\nn = {n_non_hdl}",
            f"Included in secondary analysis\nOGTT 120-min endpoint\nn = {n_ogtt120}",
        ]

        side_boxes = [
            None,
            f"Excluded\nMissing anti-TPO\nn = {excluded_missing_anti_tpo}",
            f"Excluded after TAI definition\nMissing TG and/or HDL\nn = {excluded_primary_after_tai}",
            None,
            f"Excluded after TAI definition\nMissing glucose at 120 min\nn = {excluded_ogtt_after_tai}",
        ]

        fig, ax = plt.subplots(figsize=(12, 12))
        ax.set_xlim(0, 12)
        ax.set_ylim(0, 14)
        ax.axis("off")

        main_x = 3.0
        main_w = 4.5
        main_h = 1.2
        side_x = 8.4
        side_w = 2.8
        side_h = 1.0
        ys = [12.2, 9.8, 7.4, 5.0, 2.6]

        for y, label in zip(ys, main_boxes):
            patch = FancyBboxPatch(
                (main_x, y),
                main_w,
                main_h,
                boxstyle="round,pad=0.03,rounding_size=0.05",
                linewidth=1.4,
                edgecolor="black",
                facecolor="white",
            )
            ax.add_patch(patch)
            ax.text(
                main_x + main_w / 2,
                y + main_h / 2,
                label,
                ha="center",
                va="center",
                fontsize=10,
            )

        for i in range(len(ys) - 1):
            ax.annotate(
                "",
                xy=(main_x + main_w / 2, ys[i + 1] + main_h),
                xytext=(main_x + main_w / 2, ys[i]),
                arrowprops=dict(arrowstyle="->", lw=1.4),
            )

        for y, label in zip(ys, side_boxes):
            if label is None:
                continue
            patch = FancyBboxPatch(
                (side_x, y + 0.1),
                side_w,
                side_h,
                boxstyle="round,pad=0.03,rounding_size=0.05",
                linewidth=1.2,
                edgecolor="black",
                facecolor="white",
            )
            ax.add_patch(patch)
            ax.text(
                side_x + side_w / 2,
                y + 0.1 + side_h / 2,
                label,
                ha="center",
                va="center",
                fontsize=9,
            )

            ax.annotate(
                "",
                xy=(side_x, y + 0.6),
                xytext=(main_x + main_w, y + 0.6),
                arrowprops=dict(arrowstyle="->", lw=1.2),
            )

        ax.set_title("STROBE participant flow diagram", fontsize=14, pad=20)

        flow_png = bundle_figures / "strobe_flowchart.png"
        flow_pdf = bundle_figures / "strobe_flowchart.pdf"

        fig.savefig(flow_png, dpi=300, bbox_inches="tight")
        fig.savefig(flow_pdf, bbox_inches="tight")
        plt.close(fig)

        print("Saved:", flow_png)
        print("Saved:", flow_pdf)

Saved: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle/figures/strobe_flowchart.png
Saved: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle/figures/strobe_flowchart.pdf


## Manifest

In [ ]:

manifest_rows = []

for f in sorted(bundle_dir.rglob("*")):
    if f.is_dir():
        continue
    rel = f.relative_to(bundle_dir)
    manifest_rows.append(
        {
            "relative_path": str(rel),
            "bytes": f.stat().st_size,
            "sha256": sha256_file(f),
            "modified_time": datetime.fromtimestamp(f.stat().st_mtime).isoformat(timespec="seconds"),
        }
    )

manifest = pd.DataFrame(manifest_rows).sort_values("relative_path")
out_manifest = bundle_meta / "MANIFEST_sha256.csv"
manifest.to_csv(out_manifest, index=False)

print("Saved:", out_manifest)
print(manifest.head(10))

Saved: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle/meta/MANIFEST_sha256.csv
                              relative_path  bytes  \
0        figures/F1_distributions_TAI_A.png  92188   
1         figures/F2_androgens_by_TAI_A.png  62839   
2        figures/F3_missingness_barplot.png  57369   
3        figures/F6_calibration_non_hdl.png  52907   
4        figures/F6_calibration_ogtt120.png  52136   
5        figures/F7_forest_secondary_OR.png  51281   
6  figures/F_S10_calibration_ep_non_hdl.png  73349   
7  figures/F_S10_calibration_ep_ogtt120.png  73082   
8  figures/F_S10_calibration_ep_primary.png  69639   
9           figures/F_S10_cooks_primary.png  60035   

                                              sha256        modified_time  
0  f54e066fe4a3ebc5973ba413195ce9c05a42326d4b1596...  2026-03-15T07:33:59  
1  bbb2042b3ca8ba4be3bc03041c722a53b255dd56563a59...  2026-03-15T07:33:59  
2  177f4dc21b601e21fb06c41a233c492bb57d8820dc4016...  20

## Session info

In [ ]:

session = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": getattr(np, "__version__", None),
    "pandas": getattr(pd, "__version__", None),
}


def safe_import_version(pkg: str) -> Optional[str]:
    try:
        mod = __import__(pkg)
        return getattr(mod, "__version__", None)
    except Exception:
        return None


for pkg in ["scipy", "statsmodels", "sklearn", "matplotlib", "patsy", "seaborn"]:
    session[pkg] = safe_import_version(pkg)

out_session = bundle_meta / "session_info.json"
with open(out_session, "w", encoding="utf-8") as f:
    json.dump(session, f, indent=2)

print("Saved:", out_session)
print(session)

Saved: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle/meta/session_info.json
{'timestamp': '2026-03-15T08:25:02', 'python': '3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]', 'platform': 'Linux-6.6.113+-x86_64-with-glibc2.35', 'numpy': '2.0.2', 'pandas': '2.2.2', 'scipy': '1.16.3', 'statsmodels': '0.14.6', 'sklearn': '1.6.1', 'matplotlib': '3.10.0', 'patsy': '1.0.2', 'seaborn': '0.13.2'}


## README

In [ ]:

readme = f"""\
# Export bundle: {project_name} v{version}

Created: {datetime.now().isoformat(timespec="seconds")}

## Contents
- tables/   : Manuscript and supplementary tables (CSV)
- figures/  : Figures copied from upstream steps plus STROBE flowchart generated in S11
- reports/  : Notes, logs, config snapshots, auxiliary reports
- data/     : Key analysis datasets (CSV/Parquet)
- models/   : Any serialized models or model-adjacent JSON outputs
- meta/     : MANIFEST (sha256), session_info, config snapshots, mappings

## Key compiled outputs
- tables/MASTER_results_compiled.csv
- tables/MASTER_supplement_compiled.csv
- figures/strobe_flowchart.png
- figures/strobe_flowchart.pdf
- meta/MANIFEST_sha256.csv
- meta/session_info.json

## Analysis notes
- This bundle is a packaging/export step only; no modeling is performed here.
- Reported binary endpoint models in the project use Firth logistic regression.
- Some S10 diagnostic artifacts are based on auxiliary standard logistic approximations
  and should be interpreted according to the notes saved in reports/.

## Directory policy
- Upstream artifacts from S00-S10 are copied into this bundle from configured source directories.
- New artifacts generated by S11 are written only inside the bundle and not back into upstream shared folders.

## Expected upstream steps
- Tables and figures should already exist from S00-S10.
- Particularly important upstream files include:
  * primary model tables
  * secondary model tables
  * sensitivity analysis tables
  * S09 E-value outputs
  * S10 diagnostics outputs

## Reproducibility
- File integrity hashes are provided in meta/MANIFEST_sha256.csv.
- Session/package versions are provided in meta/session_info.json.
"""

out_readme = bundle_dir / "README.txt"
with open(out_readme, "w", encoding="utf-8") as f:
    f.write(readme)

print("Saved:", out_readme)
print(readme[:800])

Saved: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle/README.txt
# Export bundle: PCOS_TAI_cross_sectional v0.2.1

Created: 2026-03-15T08:25:31

## Contents
- tables/   : Manuscript and supplementary tables (CSV)
- figures/  : Figures copied from upstream steps plus STROBE flowchart generated in S11
- reports/  : Notes, logs, config snapshots, auxiliary reports
- data/     : Key analysis datasets (CSV/Parquet)
- models/   : Any serialized models or model-adjacent JSON outputs
- meta/     : MANIFEST (sha256), session_info, config snapshots, mappings

## Key compiled outputs
- tables/MASTER_results_compiled.csv
- tables/MASTER_supplement_compiled.csv
- figures/strobe_flowchart.png
- figures/strobe_flowchart.pdf
- meta/MANIFEST_sha256.csv
- meta/session_info.json

## Analysis notes
- This bundle is a packaging/export step only; no modeling is performed her


## ZIP bundle

In [ ]:

zip_path = shutil.make_archive(str(bundle_dir), "zip", root_dir=bundle_dir)
zip_path = Path(zip_path)

print("ZIP created:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))

ZIP created: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle.zip
ZIP size (MB): 0.9


## Core output validation

In [ ]:

required_any_of = {
    "primary_model_table": ["Table2_primary_model.csv", "Table1_primary_model.csv"],
}

required_exact = [
    "Table3_secondary_models.csv",
    "TableS1_sensitivity.csv",
    "TableS2_Evalues.csv",
    "TableS10_model_diagnostics.csv",
    "TableS10_cohort_integrity_and_events.csv",
    "MASTER_results_compiled.csv",
    "MASTER_supplement_compiled.csv",
]

missing = []

for label, candidates in required_any_of.items():
    if not any((bundle_tables / fn).exists() for fn in candidates):
        missing.append(f"{label}: one of {candidates}")

for fn in required_exact:
    p = bundle_tables / fn
    if not p.exists():
        missing.append(fn)

if missing:
    print("[WARN] Missing key outputs:")
    for fn in missing:
        print(" -", fn)
else:
    print("All core tables are present in the bundle.")


[WARN] Missing key outputs:
 - primary_model_table: one of ['Table2_primary_model.csv', 'Table1_primary_model.csv']
 - TableS1_sensitivity.csv
 - TableS2_Evalues.csv


## Final summary

In [ ]:

print("\n=== S11 Summary ===")
print("Bundle dir:", bundle_dir)
print("Tables copied:", len(list(bundle_tables.glob("*"))))
print("Figures copied:", len(list(bundle_figures.glob("*"))))
print("Reports copied:", len(list(bundle_reports.glob("*"))))
print("Data copied:", len(list(bundle_data.glob("*"))))
print("Meta files:", len(list(bundle_meta.glob("*"))))
print("Models copied:", len(list(bundle_models.glob("*"))))
print("ZIP:", zip_path)


=== S11 Summary ===
Bundle dir: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle
Tables copied: 26
Figures copied: 12
Reports copied: 9
Data copied: 0
Meta files: 2
Models copied: 0
ZIP: /content/outputs/release/PCOS_TAI_cross_sectional_v0.2.1_20260315_075045_analysis_bundle.zip
